# PPO Metric Reosurce Managment

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["SDL_AUDIODRIVER"] = "dummy"

from environment.wrappers.zero_by_status import ZeroJobUsageByTheirStatus
from environment.core.jobs import JobStatus

In [ ]:
from stable_baselines3.common.callbacks import CallbackList
import gymnasium as gym

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecVideoRecorder
import wandb
from wandb.integration.sb3 import WandbCallback
import logging

logging.basicConfig(level=logging.ERROR)

In [ ]:
total_timesteps = 500_000

policy_kwargs = dict(
    net_arch=[32,64,32]
)

config = {
    "policy_type": "MultiInputPolicy",
    "total_timesteps": total_timesteps,
    "env_name": "resource-management-online",
}

In [ ]:
run = wandb.init(
    project="cluster-scheduling",
    config=config,
    sync_tensorboard=True,
    monitor_gym=True,
    save_code=True,
)

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


In [ ]:
def make_env(
    n_jobs: int,
    n_machines: int,
    n_resources: int,
    n_ticks: int,
    max_episode_steps: int = 300,
    # not_termination_penalty: float = -1e4
):
    print(f"{n_jobs=}, {n_machines=}, {n_resources=}, {n_ticks=}, {max_episode_steps=}")
    env = gym.make(
        config["env_name"],
        render_mode="rgb_array",
        n_jobs=n_jobs,
        n_machines=n_machines,
        n_resources=n_resources,
        n_ticks=n_ticks,
    )
    env = ZeroJobUsageByTheirStatus(env, JobStatus.Running, JobStatus.Completed, JobStatus.NotCreated)
    env = Monitor(env)
    return env

In [ ]:
env = DummyVecEnv([lambda: make_env(5, 1, 2, 5)])
env = VecVideoRecorder(
    env,
    f"videos/{run.id}",
    record_video_trigger=lambda x: x % 2_000 == 0,
    video_length=200,
)
model = PPO(
    config["policy_type"],
    env,
    policy_kwargs=policy_kwargs,
    verbose=1,
    tensorboard_log=f"runs/{run.id}"
)
wandb_callback = WandbCallback(
    gradient_save_freq=1_000,
    model_save_path=f"models/{run.id}",
    verbose=2,
)
# metric_callback = CustomMetricsCallback(verbose=1)
model.learn(
    total_timesteps=config["total_timesteps"],
    callback=CallbackList([
        # metric_callback,
        wandb_callback
    ]),
)

n_jobs=5, n_machines=1, n_resources=2, n_ticks=5, max_episode_steps=300
Using cpu device


/Users/guyarieli/PycharmProjects/resource_managment/.venv/lib/python3.11/site-packages/gymnasium/utils/passive_env_checker.py:29: UserWarning: WARN: It seems a Box observation space is an image but the `dtype` is not `np.uint8`, actual type: float32. If the Box observation space is not an image, we recommend flattening the observation to have only a 1D vector.
  logger.warn(
/Users/guyarieli/PycharmProjects/resource_managment/.venv/lib/python3.11/site-packages/gymnasium/utils/passive_env_checker.py:164: UserWarning: WARN: The obs returned by the `reset()` method was expecting numpy array dtype to be float32, actual type: float64
  logger.warn(
/Users/guyarieli/PycharmProjects/resource_managment/.venv/lib/python3.11/site-packages/gymnasium/utils/passive_env_checker.py:188: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")
/Users/guyarieli/PycharmProjects/resource_managment/.venv/l

KeyError: 'jobs_status'

In [ ]:
make_env(5, 3, 2, 10).observation_space, make_env(5, 3, 2, 10).action_space